# 09 OCR scoring (M2-2), one engine per fresh runtime
Runs on Colab, **fresh runtime**, CPU. Plan and scoring rules: PLAN.md (M2, decisions of 2026-11-15). Set `ENGINE` in the parameter cell; restart the runtime for the next engine. First run with `INSTALL = False` to see the package plan and wheel sizes, then set `INSTALL = True`.

- **Sample cells** (the two timing cells, 20 crops each): timings only, no score.
- **Full evaluation cells** (from `predict_text` on): written for `ENGINE = 'fastplate'` only (other engines raise `NotImplementedError`). US: all 129 crops scored with `alpr.metrics.score_us`. UC3M: all 2,547 plates, resumable, scored with the wildcard rule. Predictions go to CSVs under `reports/m2/` on Drive.

**Privacy:** plate text and label values are never printed; only counts, times and rates are.

In [ ]:
# Parameters
ENGINE = 'paddle'   # 'paddle' | 'easyocr' | 'fastplate'  (one per fresh runtime)
INSTALL = False     # False: only print the install plan and wheel sizes
SEED = 0
N_SAMPLE, REPEATS = 20, 3
DRIVE = '/content/drive/MyDrive/alpr'
LOCAL = '/content/tars'
MD5 = {   # recorded in PLAN.md
    'us_crops.zip': 'c43e2910b0042d93f5de9f0d1fca1bf2',
    'us_text_labels.csv': 'c82118867fa7256d4baab25d54537255',
    'uc3m_yolo.tar': 'e3ef306731f33095bac983338c6adb04',
    'uc3m_labels.csv': 'deccc9443b71c73fe247666d8466527a',
}
PKGS = {'paddle': 'paddleocr==3.7.0 paddlepaddle==3.3.1', 'easyocr': 'easyocr', 'fastplate': 'fast-plate-ocr[onnx]'}
FASTPLATE_MODEL = 'cct-s-v2-global-model'
# oneDNN off on CPU: paddlepaddle 3.3.x fails in the oneDNN path (PaddlePaddle issue 77340); pins stay as in uv.lock
REPORTS = f'{DRIVE}/reports/m2'
FLUSH_EVERY = 100   # UC3M predictions are appended to the CSV and flushed every 100 plates
EXPECTED = {'us': 129, 'uc3m': 2547, 'uc3m_test': 491}   # plates (PLAN.md), not images
PADDLE_KWARGS = dict(lang='en', use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False, enable_mkldnn=False)

In [ ]:
import csv, hashlib, io, itertools, json, os, platform, random, re, shutil, statistics, subprocess, sys, tarfile, time, urllib.request, zipfile
from pathlib import Path
import numpy as np
from google.colab import drive
from PIL import Image

In [ ]:
# Install plan (nothing is installed here): packages uv would add and their wheel sizes from PyPI.
subprocess.run('pip install -q uv', shell=True, check=True)  # approved raw pip: only to bootstrap uv on Colab
r = subprocess.run(f'uv pip install --system --dry-run {PKGS[ENGINE]}', shell=True, capture_output=True, text=True)
pk = re.findall(r'^\s*\+ ([A-Za-z0-9_.\-]+)==(\S+)', r.stdout + r.stderr, re.M)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
def wheel_mb(name, ver):
    try:
        files = json.load(urllib.request.urlopen(f'https://pypi.org/pypi/{name}/{ver}/json', timeout=30))['urls']
    except Exception:
        return None
    whl = [f for f in files if f['packagetype'] == 'bdist_wheel']
    pick = [f for f in whl if 'none-any' in f['filename']] or [f for f in whl if tag in f['filename'] and 'manylinux' in f['filename'] and 'x86_64' in f['filename']]
    return pick[0]['size'] / 1e6 if pick else None
sizes = {n: wheel_mb(n, v) for n, v in pk}
for n, v in pk:
    print(f'{n}=={v}: ' + (f'{sizes[n]:.1f} MB' if sizes[n] is not None else 'size unknown'))
print(f'{len(pk)} packages, known wheel sizes total {sum(s for s in sizes.values() if s):.0f} MB, unknown: {sum(s is None for s in sizes.values())}')
if not pk:
    print(r.stdout[-1500:], r.stderr[-1500:])
if ENGINE == 'paddle':
    print('PaddleOCR arguments:', PADDLE_KWARGS)

In [ ]:
assert INSTALL, 'read the plan above, then set INSTALL = True and run again'
subprocess.run('cd /content && (git -C vehicle-plate-recognition pull -q --ff-only || git clone -q https://github.com/medrees-1000/vehicle-plate-recognition.git)', shell=True, check=True)
subprocess.run(f'uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition && uv pip install --system -q {PKGS[ENGINE]}', shell=True, check=True)

In [ ]:
from alpr import ccpd, metrics, uslabels
if ENGINE == 'paddle':
    from paddleocr import PaddleOCR
elif ENGINE == 'easyocr':
    import easyocr
else:
    from fast_plate_ocr import LicensePlateRecognizer

In [ ]:
drive.mount('/content/drive')
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
print('CPU:', cpu, '| cores:', os.cpu_count(), '| python', platform.python_version(), '| engine', ENGINE)

In [ ]:
# Engine as declared in PLAN.md (decision log 2026-11-15). Model files downloaded during init are measured by mtime.
def new_mb(t0):
    skip = [Path.home() / '.cache' / d for d in ('uv', 'pip')]
    tot = 0
    for top in (Path.home() / '.paddlex', Path.home() / '.EasyOCR', Path.home() / '.cache'):
        for f in top.rglob('*') if top.exists() else []:
            if f.is_file() and not any(s in f.parents for s in skip) and f.stat().st_mtime >= t0:
                tot += f.stat().st_size
    return tot / 1e6

t0 = time.time(); t1 = time.perf_counter()
if ENGINE == 'paddle':
    eng = PaddleOCR(**PADDLE_KWARGS)
    run = lambda a: eng.predict(a[:, :, ::-1])
elif ENGINE == 'easyocr':
    eng = easyocr.Reader(['en'], gpu=False)
    run = lambda a: eng.readtext(a)
else:
    eng = LicensePlateRecognizer(FASTPLATE_MODEL)
    run = lambda a: eng.run(a)
print(f'init {time.perf_counter() - t1:.0f} s | new model files since start: {new_mb(t0):.1f} MB')

def spread(v):
    q = statistics.quantiles(v, n=4)
    return f'median {statistics.median(v) * 1000:.0f} ms | min {min(v) * 1000:.0f} | IQR {q[0] * 1000:.0f}-{q[2] * 1000:.0f} | max {max(v) * 1000:.0f}'

def time_ocr(arrs):
    run(arrs[0])   # warm-up, not counted
    ts = []
    for a in arrs:
        for _ in range(REPEATS):
            t = time.perf_counter(); run(a); ts.append(time.perf_counter() - t)
    return ts

In [ ]:
# US crops: labels MD5 is asserted on load; labels only pick which crops to time (no text is read or printed).
lab = Path(DRIVE, 'archives', 'us_text_labels.csv')
assert lab.exists(), 'us_text_labels.csv not on Drive'
assert ccpd.file_md5(lab) == MD5['us_text_labels.csv'], 'us_text_labels.csv MD5 differs from PLAN.md: stop and report'
Path(LOCAL).mkdir(exist_ok=True)
zsrc, zdst = Path(DRIVE, 'archives', 'us_crops.zip'), Path(LOCAL, 'us_crops.zip')
shutil.copyfile(zsrc, zdst)
assert ccpd.file_md5(zdst) == MD5['us_crops.zip'], 'us_crops.zip MD5 differs from PLAN.md'
with open(lab, newline='', encoding='utf-8') as f:
    readable = sorted(int(r['crop_no']) for r in csv.DictReader(f) if r['final_status'] == 'readable')
pick = random.Random(SEED).sample(readable, N_SAMPLE)
with zipfile.ZipFile(zdst) as z:
    idx = {int(r['crop_no']): r for r in csv.DictReader(io.StringIO(z.read('index.csv').decode()))}
    arrs = []
    for n in pick:
        png = z.read(f'crops/{n:04d}.png')
        assert hashlib.md5(png).hexdigest() == idx[n]['md5'], f'crop {n}: MD5 differs from index.csv'
        arrs.append(np.asarray(Image.open(io.BytesIO(png)).convert('RGB')))
print(f'{len(arrs)} readable US crops, widths px: median {statistics.median(a.shape[1] for a in arrs):.0f}')
us_t = time_ocr(arrs)
print(f'{ENGINE} on US crops: {spread(us_t)} | limit 500 ms')

In [ ]:
# UC3M crops (after the US sample): crop cost and OCR cost per plate. Only boxes are read; lp_id is never read or printed.
for fn in ('uc3m_labels.csv',):
    assert ccpd.file_md5(Path(DRIVE, 'archives', fn)) == MD5[fn], f'{fn} MD5 differs from PLAN.md'
tsrc, tdst = Path(DRIVE, 'archives', 'uc3m_yolo.tar'), Path(LOCAL, 'uc3m_yolo.tar')
t0 = time.perf_counter()
if not tdst.exists():
    shutil.copyfile(tsrc, tdst)
t1 = time.perf_counter()
assert ccpd.file_md5(tdst) == MD5['uc3m_yolo.tar'], 'uc3m_yolo.tar MD5 differs from PLAN.md'
print(f'uc3m tar: copy {t1 - t0:.0f} s, MD5 check {time.perf_counter() - t1:.0f} s')
with open(Path(DRIVE, 'archives', 'uc3m_labels.csv'), newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['split'] == 'test']
rows = random.Random(SEED).sample(rows, N_SAMPLE)
cropt, arrs2 = [], []
with tarfile.open(tdst) as t:
    by_stem = {Path(m.name).stem: m.name for m in t.getmembers() if m.isfile() and m.name.startswith('images/test/')}
    for r in rows:
        t0 = time.perf_counter()
        with Image.open(io.BytesIO(t.extractfile(by_stem[r['image_id']]).read())) as im:
            im = im.convert('RGB')
            assert im.size == (int(r['image_w']), int(r['image_h'])), 'image size differs from the labels file'
            box = uslabels.crop_box(tuple(float(r[k]) for k in ('cx', 'cy', 'w', 'h')), *im.size)
            arrs2.append(np.asarray(im.crop(box)))
        cropt.append(time.perf_counter() - t0)
print(f'UC3M crop (decode + crop) per plate: {spread(cropt)}')
print(f'crop widths px: median {statistics.median(a.shape[1] for a in arrs2):.0f}')
uc_t = time_ocr(arrs2)
print(f'{ENGINE} on UC3M crops: {spread(uc_t)} | limit 500 ms')

In [ ]:
# Predicted text for one crop. fast-plate-ocr 1.1.0: LicensePlateRecognizer.run_one(array) returns a PlatePrediction whose .plate is the decoded str
# (trailing pad characters already removed; '' if nothing was decoded). In-memory arrays must already be in the model's colour mode (this model: rgb, 64 x 128 input).
def predict_text(arr):
    if ENGINE != 'fastplate':
        raise NotImplementedError("predict_text is written for ENGINE='fastplate' only; paddle and easyocr need their own output handling (not written, not guessed)")
    return eng.run_one(arr).plate

if ENGINE == 'fastplate':
    assert eng.config.image_color_mode == 'rgb', 'the model is not an rgb model: crops would need converting'
    prov = eng.model.get_providers()   # onnxruntime may list more than one CPU-side provider
    assert 'CPUExecutionProvider' in prov and not any(x in prov for x in ('CUDAExecutionProvider', 'TensorrtExecutionProvider')), 'the timing reference is CPU'

In [ ]:
# US part: all US crops -> predictions CSV on Drive -> metrics.score_us. Prints counts, times and rates only; plate text is never printed.
assert ccpd.file_md5(lab) == MD5['us_text_labels.csv'], 'us_text_labels.csv MD5 differs from PLAN.md: stop and report'
Path(REPORTS).mkdir(parents=True, exist_ok=True)
with open(lab, newline='', encoding='utf-8') as f:
    labels = {int(r['crop_no']): r for r in csv.DictReader(f)}
preds, t_crop, t_ocr = {}, 0.0, 0.0
with zipfile.ZipFile(zdst) as z:
    idx = {int(r['crop_no']): r for r in csv.DictReader(io.StringIO(z.read('index.csv').decode()))}
    assert set(idx) == set(labels) and len(idx) == EXPECTED['us'], 'crop numbers or count differ from PLAN.md'
    for n in sorted(idx):
        t0 = time.perf_counter()
        png = z.read(f'crops/{n:04d}.png')
        assert hashlib.md5(png).hexdigest() == idx[n]['md5'], f'crop {n}: MD5 differs from index.csv'
        arr = np.asarray(Image.open(io.BytesIO(png)).convert('RGB'))
        t1 = time.perf_counter()
        preds[n] = predict_text(arr)
        t_crop += t1 - t0
        t_ocr += time.perf_counter() - t1
out = Path(REPORTS, f'us_pred_{ENGINE}.csv')
part = out.with_name(out.name + '.part')
with open(part, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['crop_no', 'engine', 'pred'])
    w.writerows([n, ENGINE, preds[n]] for n in sorted(preds))
part.replace(out)
rows = [{'pred': preds[n], 'proposed_text': r['proposed_text'], 'corrected_text': r['corrected_text'], 'final_status': r['final_status'], 'reason': r['reason']}
        for n, r in sorted(labels.items())]
sc = metrics.score_us(rows)
print(f'US {ENGINE}: {len(preds)} crops | crop step {t_crop:.1f} s | OCR step {t_ocr:.1f} s ({t_ocr / len(preds) * 1000:.0f} ms per crop) | excluded {sc["excluded"]}')
for g in ('confirmed', 'confirmed_no_partial', 'confirmed_plus_guess', 'partial'):
    d = sc[g]
    print(f'  {g}: n {d["n"]} | exact {d["exact"]} ({"n/a" if d["exact_rate"] is None else format(d["exact_rate"], ".4f")}) | CER {"n/a" if d["cer"] is None else format(d["cer"], ".4f")}')

In [ ]:
# UC3M part: all plates (not images), streamed image by image. Predictions are appended to a CSV on Drive and flushed every FLUSH_EVERY plates;
# a rerun skips plates already in the file. The labels file has a split column, so the authors' test plates are those with split 'test'.
assert ccpd.file_md5(Path(DRIVE, 'archives', 'uc3m_labels.csv')) == MD5['uc3m_labels.csv'], 'uc3m_labels.csv MD5 differs from PLAN.md'
with open(Path(DRIVE, 'archives', 'uc3m_labels.csv'), newline='', encoding='utf-8') as f:
    plates = sorted(csv.DictReader(f), key=lambda r: (r['split'], r['image_id'], int(r['plate_index'])))
assert len(plates) == EXPECTED['uc3m'] and sum(r['split'] == 'test' for r in plates) == EXPECTED['uc3m_test'], 'plate counts differ from PLAN.md'
pkey = lambda r: f"{r['split']}/{r['image_id']}/{r['plate_index']}"
out = Path(REPORTS, f'uc3m_pred_{ENGINE}.csv')
done = {}
if out.exists():   # rows with a wrong number of fields (a write cut by a disconnect) are ignored and redone; a repeated key keeps the last row
    with open(out, newline='', encoding='utf-8') as f:
        for r in csv.reader(f):
            if len(r) == 3 and r[0] != 'key':
                done[r[0]] = r[2]
todo = [r for r in plates if pkey(r) not in done]
print(f'UC3M {ENGINE}: {len(done)} plates already in the CSV, {len(todo)} to do')
t_crop, t_ocr, n_new = 0.0, 0.0, 0
if todo:
    with tarfile.open(tdst) as t, open(out, 'a', newline='', encoding='utf-8') as f:
        member = {(n.split('/')[1], Path(n).stem): n for n in (m.name for m in t.getmembers() if m.isfile()) if n.startswith('images/')}
        w = csv.writer(f)
        if not done:
            w.writerow(['key', 'engine', 'pred'])
        for (split, image_id), grp in itertools.groupby(todo, key=lambda r: (r['split'], r['image_id'])):
            t0 = time.perf_counter()
            with Image.open(io.BytesIO(t.extractfile(member[(split, image_id)]).read())) as im:
                im = im.convert('RGB')
                crops = []
                for r in grp:
                    assert im.size == (int(r['image_w']), int(r['image_h'])), 'image size differs from the labels file'
                    crops.append((pkey(r), np.asarray(im.crop(uslabels.crop_box(tuple(float(r[k]) for k in ('cx', 'cy', 'w', 'h')), *im.size)))))
            t_crop += time.perf_counter() - t0
            for key, arr in crops:
                t1 = time.perf_counter()
                w.writerow([key, ENGINE, predict_text(arr)])
                t_ocr += time.perf_counter() - t1
                n_new += 1
                if n_new % FLUSH_EVERY == 0:
                    f.flush(); os.fsync(f.fileno())
                if n_new % 500 == 0:
                    print(f'  {n_new} of {len(todo)} done | crop step {t_crop:.0f} s | OCR step {t_ocr:.0f} s')
        f.flush(); os.fsync(f.fileno())
    print(f'this run: {n_new} plates | crop step (decode + crop) {t_crop:.0f} s | OCR step {t_ocr:.0f} s ({t_ocr / max(n_new, 1) * 1000:.0f} ms per plate)')

In [ ]:
# UC3M scoring with the wildcard rule (PLAN.md, UC3M-LP rule c): exact-match accuracy over all plates and over the authors' test plates. No CER (declared for US only).
done = {}
with open(out, newline='', encoding='utf-8') as f:
    for r in csv.reader(f):
        if len(r) == 3 and r[0] != 'key':
            done[r[0]] = r[2]
if any(pkey(r) not in done for r in plates):
    print(f'incomplete: {sum(pkey(r) in done for r in plates)} of {len(plates)} plates have a prediction; rerun the previous cell, no score yet')
else:
    hit = {pkey(r): metrics.wildcard_match(done[pkey(r)], r['lp_id']) for r in plates}
    for name, sel in (('all plates', plates), ("authors' test split", [r for r in plates if r['split'] == 'test'])):
        k = sum(hit[pkey(r)] for r in sel)
        print(f'UC3M {ENGINE}, {name}: n {len(sel)} | wildcard exact match {k} ({k / len(sel):.4f})')